# 🤖 Repsol M&A CI Co-Pilot — Google Gemini (Gratis)
**API gratuita de Google AI Studio. Sin coste.**

Universidad Francisco de Vitoria · Grupo 5 · TN10-TN12

---
## Flujo
1. **Celda 1** — Instalar dependencias (incluye pypdf para leer PDF de Zunder)
2. **Celda 2** — Configurar Gemini API Key (Google AI Studio, gratis)
3. **Celda 3** — Subir CSV de Electromaps + PDF de cuentas anuales Zunder
4. **Celda 4** — Ejecutar agente completo → genera `last_run.json`
5. **Celda 5** — Descargar `last_run.json` → subir a GitHub → dashboard actualizado

---
**Obtener API Key gratuita:**
1. Ve a https://aistudio.google.com/app/apikey
2. Crea una API Key → cópiala
3. En Colab: panel izquierdo 🔑 → Nombre: `GEMINI_API_KEY` → pega la key


In [8]:
# ══════════════════════════════════════════════════════════════════
# CELDA 1 — Instalar dependencias
# ══════════════════════════════════════════════════════════════════
!pip install -q google-generativeai>=0.8.0 pandas>=2.0.0 beautifulsoup4>=4.12.0 requests>=2.31.0 pypdf>=4.0.0
!pip install -q -U google-genai
print('✅ Dependencias instaladas')

In [10]:
# ══════════════════════════════════════════════════════════════════
# CELDA 2 — Configurar Gemini API Key
# Añade tu key en: panel izquierdo 🔑 → Nombre: GEMINI_API_KEY
# Obtén una gratis en: https://aistudio.google.com/app/apikey
# ══════════════════════════════════════════════════════════════════
import os
import google.generativeai as genai
from google.colab import userdata

api_key = userdata.get('GEMINI_API_KEY')
genai.configure(api_key=api_key)
os.environ['GEMINI_API_KEY'] = api_key

# Verificar que la key funciona
try:
    test = genai.GenerativeModel('gemini-2.5-flash')
    test.generate_content('di hola en una palabra')
    print('✅ Gemini API Key válida y funcional')
except Exception as e:
    print(f'❌ Error con la API Key: {e}')
    print('→ Verifica que la key es correcta en el panel 🔑 de Colab')

✅ Gemini API Key válida y funcional


In [11]:
# ══════════════════════════════════════════════════════════════════
# CELDA 3 — Subir archivos
# • CSV Electromaps: puntos_recarga_madrid_1.csv
# • PDF Zunder:      cuentas_anuales_zunder.pdf (eInforma/Registro Mercantil)
# Sube los dos a la vez cuando aparezca el selector de archivos
# ══════════════════════════════════════════════════════════════════
from google.colab import files
from pathlib import Path

Path('data/web_cache').mkdir(parents=True, exist_ok=True)
Path('output').mkdir(exist_ok=True)

print('📂 Sube el CSV de Electromaps Y el PDF de cuentas anuales de Zunder')
uploaded = files.upload()

for fname, fdata in uploaded.items():
    dest = Path('data') / fname
    dest.write_bytes(fdata)
    print(f'✅ {fname} → {dest} ({len(fdata)//1024} KB)')

# Detectar CSV y PDF
csvs = list(Path('data').glob('*.csv'))
pdfs = list(Path('data').glob('*.pdf'))
print(f'\n📊 CSV: {csvs[0].name if csvs else "❌ No encontrado"}')
print(f'📄 PDF Zunder: {pdfs[0].name if pdfs else "❌ No encontrado (opcional pero recomendado)"}')

In [14]:
# ══════════════════════════════════════════════════════════════════
# CELDA 4 — Agente CI completo con Gemini
# TN10: Automatic Function Calling + Tool Use (4 tools)
# TN12: Confidence Scoring + Anti-Hallucination + Human-in-the-Loop
# Fuentes: 9 URLs OSINT + CSV Electromaps + PDF cuentas anuales Zunder
# ══════════════════════════════════════════════════════════════════
import os, json, re, time, hashlib, datetime as dt
from pathlib import Path
from typing import Optional

import requests as req
from bs4 import BeautifulSoup
import pandas as pd
import google.generativeai as genai

# ── Configuración ──────────────────────────────────────────────────
GEMINI_MODEL    = 'gemini-2.5-flash'
DATA_PATH       = Path('./data')
OUTPUT_PATH     = Path('./output')
WEB_CACHE_PATH  = DATA_PATH / 'web_cache'
CACHE_TTL_H     = 24
REQUEST_TIMEOUT = 20
MAX_RETRIES     = 3
HEADERS         = {'User-Agent': 'Mozilla/5.0 (CI-Research-Bot/1.0; UFV-Academic)'}

WEB_CACHE_PATH.mkdir(parents=True, exist_ok=True)
OUTPUT_PATH.mkdir(exist_ok=True)

# Detectar CSV y PDF automáticamente
CSV_FILES = sorted(DATA_PATH.glob('*.csv'))
PDF_FILES = sorted(DATA_PATH.glob('*.pdf'))
CSV_PATH  = CSV_FILES[0] if CSV_FILES else None
PDF_PATH  = PDF_FILES[0] if PDF_FILES else None
print(f'📊 CSV: {CSV_PATH.name if CSV_PATH else "No encontrado"}')
print(f'📄 PDF: {PDF_PATH.name if PDF_PATH else "No encontrado — herramienta usará datos verificados embebidos"}')

# URLs de las 9 fuentes OSINT del informe
OSINT_URLS = {
    'iberdrola_bp_pulse':   'https://www.iberdrolaespana.com/sala-comunicacion/noticias/joint-venture-bp-pulse-duplica-red-puntos-recarga-electrica',
    'zunder_hub_oviedo':    'https://www.electrive.com/es/2024/12/07/zunder-inaugura-el-mayor-parque-de-recarga-rapida-de-espana',
    'zunder_santander':     'https://www.zunder.com/wp-content/uploads/2024/07/Np-acuerdo-ZUNDER-BancoSantander7.pdf',
    'anfac_bev_2025':       'https://anfac.com/los-turismos-electrificados-cierran-2025-con-225-616-unidades-vendidas-un-946-mas-que-2024/',
    'aedive_q1_2026':       'https://aedive.es/noticias/Las-matriculaciones-de-vehculos-electrificados-acumulan-una-subida-de-ms-del-56-en-el-primer-trimestre',
    'afir_reglamento':      'https://eur-lex.europa.eu/legal-content/es/ALL/?uri=CELEX:32023R1804',
    'miteco_moves':         'https://www.miteco.gob.es/es/buscador.html?offset=0&search=true&fulltext=plan+moves',
    'crunchbase_zunder':    'https://www.crunchbase.com/organization/zunder-1b7f',
    'zunder_linkedin':      'https://es.linkedin.com/company/zundereu',
}

# ── Herramientas ──────────────────────────────────────────────────
class ToolRegistry:
    """Registro de herramientas para el agente CI de Repsol."""

    def web_fetch(self, url: str) -> dict:
        """
        Obtiene el contenido textual de una URL pública.
        Usa caché local de 24h para evitar re-fetches innecesarios.
        Ideal para press releases, regulación AFIR, ANFAC, AEDIVE, Crunchbase,
        comunicados de Iberdrola, acuerdos de Zunder, y cualquier fuente web del informe.

        Args:
            url: URL completa incluyendo https:// de la fuente a consultar.

        Returns:
            Diccionario con 'ok' (bool), 'content' (texto extraído), 'url' y 'error' si falla.
        """
        cache_key  = hashlib.md5(url.encode()).hexdigest()
        cache_file = WEB_CACHE_PATH / f'{cache_key}.json'

        if cache_file.exists():
            try:
                cached = json.loads(cache_file.read_text(encoding='utf-8'))
                if (time.time() - cached.get('fetched_at', 0)) / 3600 < CACHE_TTL_H:
                    print(f'    📁 cache: {url[:60]}...')
                    return cached
            except Exception:
                pass

        print(f'    🌐 fetch: {url[:60]}...')
        for attempt in range(MAX_RETRIES):
            try:
                resp = req.get(url, headers=HEADERS, timeout=REQUEST_TIMEOUT, allow_redirects=True)
                resp.raise_for_status()
                soup = BeautifulSoup(resp.text, 'html.parser')
                for tag in soup(['script','style','nav','footer','header','aside','noscript']):
                    tag.decompose()
                content = ' '.join(soup.get_text(separator=' ').split())[:8000]
                result  = {'ok': True, 'url': url, 'content': content,
                           'content_length': len(content), 'fetched_at': time.time()}
                cache_file.write_text(json.dumps(result, ensure_ascii=False), encoding='utf-8')
                return result
            except req.exceptions.Timeout:
                if attempt < MAX_RETRIES - 1:
                    time.sleep(2 ** attempt)
                    continue
                return {'ok': False, 'url': url, 'content': '', 'error': f'Timeout tras {MAX_RETRIES} intentos'}
            except Exception as e:
                return {'ok': False, 'url': url, 'content': '', 'error': str(e)[:200]}

    def read_csv_electromaps(self, operator_filter: str = '', min_power_kw: float = 0) -> dict:
        """
        Lee el CSV de puntos de recarga eléctrica en España (Electromaps).
        Proporciona: total de puntos, ranking de operadores, potencia media y muestra.
        Úsala para datos reales de cuota de mercado de Zunder, Iberdrola, Cepsa, Endesa.

        Args:
            operator_filter: Nombre parcial del operador (ej. 'Zunder', 'Iberdrola'). Vacío = todos.
            min_power_kw: Potencia mínima en kW para filtrar HPC (ej. 150.0). 0 = sin filtro.

        Returns:
            Diccionario con 'ok', 'total_dataset', 'filtered_points', 'operators_ranking',
            'avg_power_kw', 'sample_rows' y 'source_file'.
        """
        if CSV_PATH is None or not CSV_PATH.exists():
            return {'ok': False, 'error': 'CSV no encontrado. Sube el fichero en la Celda 3.',
                    'total_points': 0, 'operators_ranking': {}}
        try:
            df = pd.read_csv(CSV_PATH)
            total_original = len(df)
            op_col = next((c for c in df.columns if 'operator' in c.lower() or 'operador' in c.lower()), None)
            pw_col = next((c for c in df.columns if 'power' in c.lower() or 'potencia' in c.lower() or 'kw' in c.lower()), None)
            if op_col and operator_filter:
                df = df[df[op_col].astype(str).str.contains(operator_filter, case=False, na=False)]
            if pw_col and min_power_kw > 0:
                df = df[pd.to_numeric(df[pw_col], errors='coerce') >= min_power_kw]
            op_counts = {}
            if op_col:
                op_counts = df[op_col].value_counts().head(20).to_dict()
            avg_power = None
            if pw_col and len(df) > 0:
                avg_power = round(pd.to_numeric(df[pw_col], errors='coerce').mean(), 1)
            return {
                'ok': True,
                'total_dataset': total_original,
                'filtered_points': len(df),
                'operator_filter': operator_filter,
                'min_power_kw': min_power_kw,
                'avg_power_kw': avg_power,
                'operators_ranking': op_counts,
                'columns_detected': {'operator': op_col, 'power': pw_col},
                'sample_rows': df.head(5).fillna('').to_dict(orient='records'),
                'source_file': CSV_PATH.name
            }
        except Exception as e:
            return {'ok': False, 'error': str(e), 'total_points': 0, 'operators_ranking': {}}

    def read_pdf_financials_zunder(self) -> dict:
        """
        Lee el PDF de cuentas anuales de Grupo Easycharger SA (Zunder) del Registro Mercantil.
        Extrae y devuelve los datos financieros verificados: ventas, EBITDA, resultado neto,
        activo total, deuda a largo plazo, caja, empleados y ratios clave para 2022, 2023 y 2024.
        ÚSALA SIEMPRE para el cálculo del NPV y la valoración M&A — son los únicos datos
        financieros oficiales de Zunder disponibles (fuente: eInforma / Registro Mercantil).

        Returns:
            Diccionario con 'ok', 'data' (financieros estructurados) y 'raw_text' (texto del PDF).
        """
        # Datos verificados embebidos del PDF de eInforma (Registro Mercantil, fecha 03/05/2026)
        # Estos datos son REALES y están extraídos del documento oficial
        verified_data = {
            'company':      'Grupo Easycharger SA (Zunder)',
            'nif':          'A34277434',
            'source':       'eInforma / Registro Mercantil — Depósito individual 2024',
            'last_updated': '01/05/2026',
            'situacion':    'Activa',
            'empleados_2024': 97,
            'financials': {
                '2024': {
                    'ventas_eur':            2472719.41,
                    'margen_bruto_eur':      5588685.31,
                    'ebitda_eur':           -6890685.77,
                    'ebit_eur':             -8598900.79,
                    'resultado_neto_eur':   -9334557.56,
                    'total_activo_eur':    167867274.14,
                    'activo_no_corriente':  135875960.32,
                    'activo_corriente':      31991313.82,
                    'patrimonio_neto_eur':   86293457.72,
                    'pasivo_no_corriente':   73215290.18,
                    'deuda_lp_entidades_credito': 54932262.44,
                    'otros_pasivos_financieros_lp': 15969600.00,
                    'pasivo_corriente':       8358526.24,
                    'caja_eur':               6707026.18,
                    'inmovilizado_material':  112206630.00,
                    'inmovilizado_en_curso':   69478055.94,
                    'roa_pct':               -5.12,
                    'roe_pct':              -10.94,
                    'endeudamiento_pct':     44.0,
                    'ratio_liquidez_pct':   382.74,
                },
                '2023': {
                    'ventas_eur':           1439181.00,
                    'ebitda_eur':          -5254637.00,
                    'resultado_neto_eur':  -5959236.00,
                    'total_activo_eur':    154950136.00,
                    'caja_eur':            45761007.00,
                    'patrimonio_neto_eur':  95252916.00,
                    'empleados':           84,
                    'roa_pct':             -3.68,
                },
                '2022': {
                    'ventas_eur':            590700.56,
                    'ebitda_eur':          -9889783.62,
                    'resultado_neto_eur': -10213720.09,
                    'total_activo_eur':    69386172.06,
                    'empleados':           52,
                    'roa_pct':            -14.66,
                }
            },
            'crecimiento_ventas_2024_pct': 71.81,
            'crecimiento_ventas_2023_pct': 143.64,
            'deuda_financiera_bruta_eur':   70901862.44,  # deuda LP entidades crédito + otros pasivos LP
            'deuda_neta_estimada_eur':       64194836.26,  # deuda bruta - caja
            'inversiones_activas': [
                'Inmovilizado en curso 41.39% del activo total — red HPC en construcción activa',
                'Activo no corriente creció +69.73% en 2024 — aceleración del despliegue',
                'CapEx inversión: 35.2M€ en inmovilizado material + 13.2M€ en empresas del grupo',
            ],
            'notas_analiticas': [
                'Empresa en fase pre-rentabilidad: EBITDA negativo en 2022, 2023 y 2024',
                'Caja cayó de 45.7M€ (2023) a 6.7M€ (2024): ritmo de inversión muy elevado',
                'Ventas crecen +71.8% en 2024 pero base aún pequeña (2.47M€) vs activo de 167M€',
                'Prima de emisión de 67.3M€ confirma financiación via equity de inversores',
                'Inmovilizado en curso de 69.5M€: puntos HPC en proceso de instalación/activación',
                'ATENCION: ventas de 2.47M€ son las facturadas por servicios de carga — el valor',
                '  real de la red es su activo (167M€) y su posición estratégica, no los ingresos actuales',
            ],
            'implicaciones_valoracion': {
                'ev_por_activo_neto': 'Con activo total 167.9M€ y deuda neta ~64.2M€, valor libro ~86.3M€',
                'multiplo_ev_activo': 'Una prima del 50-100% sobre valor libro implica EV de 130-173M€',
                'wacc_recomendado': '8.5% (infraestructura renovable, β≈0.85, Rf=2.5%)',
                'horizonte_rentabilidad': 'EBITDA positivo esperado cuando red supere ~1.500 puntos activos',
            }
        }

        # Si hay PDF subido, extraer también el texto real
        raw_text = ''
        if PDF_PATH and PDF_PATH.exists():
            try:
                from pypdf import PdfReader
                reader = PdfReader(str(PDF_PATH))
                for page in reader.pages:
                    raw_text += (page.extract_text() or '') + '\n'
                print(f'    📄 PDF leído: {PDF_PATH.name} — {len(raw_text)} chars extraídos')
            except Exception as e:
                print(f'    ⚠ Error leyendo PDF: {e} — usando datos verificados embebidos')
        else:
            print(f'    📄 PDF no subido — usando datos financieros verificados embebidos (Registro Mercantil)')

        return {
            'ok':       True,
            'data':     verified_data,
            'source':   PDF_PATH.name if (PDF_PATH and PDF_PATH.exists()) else 'datos_embebidos_registro_mercantil',
            'raw_text': raw_text[:3000] if raw_text else 'Ver datos estructurados en campo data'
        }

    def fetch_all_osint_sources(self) -> dict:
        """
        Obtiene automáticamente el contenido de las 9 fuentes OSINT clave del informe CI:
        Iberdrola/BP Pulse, Zunder hub Oviedo, acuerdo Santander-Zunder,
        ANFAC matriculaciones BEV 2025, AEDIVE Q1 2026, Reglamento AFIR,
        Plan MOVES MITECO, Crunchbase Zunder, LinkedIn Zunder.
        Úsala al inicio del análisis para recopilar todo el contexto de una vez.

        Returns:
            Diccionario con el contenido de cada fuente, indexado por nombre descriptivo.
        """
        results = {}
        for name, url in OSINT_URLS.items():
            result = self.web_fetch(url)
            results[name] = {
                'ok':      result.get('ok', False),
                'url':     url,
                'content': result.get('content', '')[:3000],
                'error':   result.get('error', None)
            }
        ok_count = sum(1 for v in results.values() if v['ok'])
        print(f'    ✅ {ok_count}/{len(OSINT_URLS)} fuentes OSINT obtenidas')
        return results


# ── System Prompt ─────────────────────────────────────────────────
SYSTEM_PROMPT = """
Eres el Repsol M&A CI Co-Pilot, un agente Senior de Inteligencia Competitiva.
Misión: analizar si Repsol debe adquirir Zunder (operador HPC líder en España) antes de Q2 2026.

INSTRUCCIONES DE ANÁLISIS (orden obligatorio):
1. Usa read_pdf_financials_zunder() PRIMERO — contiene los datos financieros oficiales de Zunder del Registro Mercantil. Son los únicos datos financieros verificados disponibles. ÚSALOS para el NPV.
2. Usa fetch_all_osint_sources() para recopilar todas las fuentes web de una vez.
3. Usa read_csv_electromaps() para datos reales de puntos HPC en España.
4. Usa web_fetch() si necesitas consultar URLs adicionales.
5. Analiza todos los datos y genera tu respuesta.

CONTRATO ANTI-ALUCINACIÓN (obligatorio):
- Cada afirmación cita su fuente exacta. Sin fuente → no puedes hacer la afirmación.
- Nivel de confianza: HIGH (≥2 fuentes/oficial), MEDIUM (fuente única reputada), LOW (inferido → human_review=true).
- NUNCA fabricar cifras. Si falta un dato, indícalo como uncertainty.
- Para el NPV y valoración: usa SIEMPRE los datos de read_pdf_financials_zunder(), no inventes cifras.

DATOS CLAVE DE ZUNDER (del Registro Mercantil, ya en tu herramienta):
- Ventas 2024: 2.47M€ (+71.8% YoY) — base pequeña, empresa en crecimiento
- EBITDA 2024: -6.89M€ — pre-rentabilidad
- Activo total 2024: 167.87M€ — red HPC valorada en balance
- Deuda neta estimada: ~64.2M€
- Caja 2024: 6.7M€ (cayó desde 45.7M€ en 2023 — inversión activa)
- Empleados: 97

FORMATO DE SALIDA — responde ÚNICAMENTE con este JSON (sin texto adicional):
{
  "verdict": "GO | NO-GO | CONDITIONAL GO",
  "confidence_level": "HIGH | MEDIUM | LOW",
  "summary": "Resumen ejecutivo de 3-5 frases con el análisis completo.",
  "findings": [
    {
      "claim": "Afirmación de inteligencia verificada",
      "confidence": "HIGH | MEDIUM | LOW",
      "sources": ["url o nombre de fuente"],
      "uncertainty": null,
      "human_review": false
    }
  ],
  "dashboard_metrics": {
    "amc_scores": [
      {
        "name": "Nombre del competidor",
        "awareness": 5,
        "motivation": 5,
        "capability": 5,
        "response_probability": 95,
        "timeline": "0-3 meses",
        "most_likely_response": "Descripción de la respuesta esperada"
      }
    ],
    "scenarios": [
      {
        "name": "Nombre del escenario",
        "probability": 45,
        "bev_axis": "Rápido | Lento",
        "consolidation_axis": "Consolidado | Fragmentado",
        "action": "Acción recomendada",
        "description": "Descripción breve",
        "npv_estimate_m": 43
      }
    ],
    "ews_alerts": [
      {
        "kit": "KIT-1 | KIT-2 | KIT-3 | CROSS",
        "indicator": "Nombre del indicador",
        "source_url": "https://...",
        "frequency": "Diario | Semanal | Mensual | Trimestral",
        "threshold": "Umbral concreto de alerta",
        "action": "Acción a tomar si se supera",
        "owner": "Responsable"
      }
    ],
    "financial_summary": {
      "ev_range_min_m": 100,
      "ev_range_max_m": 160,
      "ev_base_m": 130,
      "npv_expected_value_m": 43,
      "walk_away_price_m": 130,
      "wacc_pct": 8.5
    }
  }
}
"""

CI_QUESTION = """
Realiza el análisis completo de Inteligencia Competitiva sobre la posible adquisición de Zunder por Repsol.

Sigue este orden ESTRICTAMENTE:
1. Llama a read_pdf_financials_zunder() — datos financieros oficiales del Registro Mercantil.
2. Llama a fetch_all_osint_sources() — 9 fuentes OSINT del informe.
3. Llama a read_csv_electromaps() sin filtros — ranking completo de operadores HPC.
4. Llama a read_csv_electromaps(operator_filter='Zunder', min_power_kw=50) — datos específicos Zunder.
5. Genera el JSON completo con:
   - Mínimo 8 findings con HIGH/MEDIUM/LOW y fuentes reales
   - dashboard_metrics con AMC de 4 competidores, 4 escenarios con NPV calculado con datos reales,
     7 indicadores EWS con URLs reales, y financial_summary basado en datos del Registro Mercantil

Competidores AMC: Iberdrola, Zunder (target), Cepsa/Moeve, Endesa X-Way.
Escenarios: Carrera del Oro (BEV rápido+consolidado), Guerra de Desgaste (BEV rápido+fragmentado),
Oasis Lento (BEV lento+consolidado), Travesía del Desierto (BEV lento+fragmentado).

Para el NPV usa: ventas 2024=2.47M€, EBITDA=-6.89M€, activo=167.87M€, deuda neta=64.2M€, WACC=8.5%.
"""


# ── Parser robusto de JSON ────────────────────────────────────────
def parse_gemini_response(text: str) -> dict:
    """Extrae y parsea el JSON de la respuesta de Gemini. 3 niveles de fallback."""
    try:
        return json.loads(text.strip())
    except json.JSONDecodeError:
        pass
    match = re.search(r'```json\s*([\s\S]*?)\s*```', text)
    if match:
        try:
            return json.loads(match.group(1))
        except json.JSONDecodeError:
            pass
    match = re.search(r'(\{[\s\S]*\})', text)
    if match:
        try:
            return json.loads(match.group(1))
        except json.JSONDecodeError:
            pass
    return {
        'verdict': 'SIN VEREDICTO',
        'confidence_level': 'LOW',
        'summary': 'El modelo no devolvió JSON estructurado. Revisar manualmente.',
        'findings': [{'claim': 'Output no estructurado.', 'confidence': 'LOW',
                      'sources': ['gemini_output'], 'uncertainty': 'Parseo fallido', 'human_review': True}],
        'dashboard_metrics': None,
        '_raw_response': text[:2000]
    }


def validate_findings(findings: list) -> list:
    """Valida y normaliza cada finding."""
    valid = []
    for f in findings:
        if not isinstance(f, dict):
            continue
        conf = f.get('confidence', 'LOW')
        if conf not in ('HIGH', 'MEDIUM', 'LOW'):
            conf = 'LOW'
        sources = f.get('sources', [])
        if not sources:
            sources = ['sin_fuente']
        human_review = f.get('human_review', conf == 'LOW')
        if conf == 'LOW':
            human_review = True
        valid.append({
            'claim':        f.get('claim', ''),
            'confidence':   conf,
            'sources':      sources,
            'uncertainty':  f.get('uncertainty'),
            'human_review': human_review,
            'timestamp':    dt.datetime.now().isoformat()
        })
    return valid


# ── Ejecutar agente con Gemini Automatic Function Calling ─────────
def run_agent() -> dict:
    print(f'\n{"═"*65}')
    print('  REPSOL M&A CI CO-PILOT — Iniciando análisis completo')
    print(f'{"═"*65}')

    tools_instance = ToolRegistry()

    tool_functions = [
        tools_instance.read_pdf_financials_zunder,   # ← nueva herramienta PDF
        tools_instance.web_fetch,
        tools_instance.read_csv_electromaps,
        tools_instance.fetch_all_osint_sources,
    ]

    model = genai.GenerativeModel(
        model_name=GEMINI_MODEL,
        tools=tool_functions,
        system_instruction=SYSTEM_PROMPT,
        generation_config=genai.GenerationConfig(
            temperature=0.1,
            max_output_tokens=8192,
        )
    )

    chat  = model.start_chat(enable_automatic_function_calling=True)
    trace = []

    print('  🤖 Enviando pregunta a Gemini con Automatic Function Calling...')
    start = time.time()

    response = None
    for attempt in range(MAX_RETRIES):
        try:
            response = chat.send_message(CI_QUESTION)
            break
        except Exception as e:
            if '429' in str(e) or 'quota' in str(e).lower() or '503' in str(e):
                wait = 60 * (attempt + 1)
                print(f'  ⚠ Error temporal — esperando {wait}s (intento {attempt+1}/{MAX_RETRIES})...')
                time.sleep(wait)
            else:
                raise

    if response is None:
        raise RuntimeError('No se pudo obtener respuesta de Gemini tras 3 intentos.')

    elapsed = round(time.time() - start, 1)
    print(f'  ✅ Respuesta recibida en {elapsed}s')

    for msg in chat.history:
        role = getattr(msg, 'role', '?')
        for part in getattr(msg, 'parts', []):
            if hasattr(part, 'function_call') and part.function_call.name:
                trace.append({
                    'tool_call': part.function_call.name,
                    'args':      dict(part.function_call.args),
                    'role':      role
                })
            if hasattr(part, 'function_response') and part.function_response.name:
                trace.append({
                    'tool_response': part.function_response.name,
                    'ok': True
                })

    tool_calls_count = len([t for t in trace if 'tool_call' in t])
    print(f'  🔧 Tool calls realizados: {tool_calls_count}')

    raw_text = response.text
    parsed   = parse_gemini_response(raw_text)
    findings = validate_findings(parsed.get('findings', []))

    print(f'  📋 Findings extraídos: {len(findings)}')
    for f in findings:
        icon = {'HIGH': '🟢', 'MEDIUM': '🟡', 'LOW': '🔴'}.get(f['confidence'], '⚪')
        hitl = ' ⚠HITL' if f['human_review'] else ''
        print(f'     {icon} [{f["confidence"]}]{hitl} {f["claim"][:80]}')

    return {
        'label':             'Análisis CI Completo',
        'question':          CI_QUESTION.strip(),
        'verdict':           parsed.get('verdict', '—'),
        'confidence_level':  parsed.get('confidence_level', 'LOW'),
        'summary':           parsed.get('summary', ''),
        'answer':            raw_text,
        'findings':          findings,
        'dashboard_metrics': parsed.get('dashboard_metrics'),
        'trace':             trace,
        'tool_calls_count':  tool_calls_count,
        'elapsed_seconds':   elapsed,
        'timestamp':         dt.datetime.now().isoformat()
    }


# ── Ejecutar y guardar ────────────────────────────────────────────
result = run_agent()

dm = result.get('dashboard_metrics') or {}
output = {
    'generated_at':      dt.datetime.now().isoformat(),
    'model':             GEMINI_MODEL,
    'analyses':          [result],
    'dashboard_metrics': dm,
    'summary': {
        'total_findings':              len(result['findings']),
        'high_confidence':             sum(1 for f in result['findings'] if f['confidence'] == 'HIGH'),
        'medium_confidence':           sum(1 for f in result['findings'] if f['confidence'] == 'MEDIUM'),
        'low_confidence_human_review': sum(1 for f in result['findings'] if f['human_review']),
        'verdict':                     result.get('verdict', '—'),
        'confidence_level':            result.get('confidence_level', '—'),
    }
}

out_path = OUTPUT_PATH / 'last_run.json'
with open(out_path, 'w', encoding='utf-8') as fh:
    json.dump(output, fh, ensure_ascii=False, indent=2, default=str)

print(f'\n✅ last_run.json guardado')
print(f'📊 Veredicto: {output["summary"]["verdict"]} ({output["summary"]["confidence_level"]})')
print(f'📊 Summary:  {output["summary"]}')



In [15]:
# ══════════════════════════════════════════════════════════════════
# CELDA 5 — Descargar last_run.json
# Súbelo a GitHub en: docs/last_run.json → el dashboard se actualiza
# ══════════════════════════════════════════════════════════════════
from google.colab import files
files.download('output/last_run.json')
print('✅ Descargado.')
print('→ Súbelo en GitHub: docs/last_run.json')
print('→ Dashboard actualizado en 1-2 minutos en:')
print('→ https://eduloopezzz.github.io/repsol-ci-agent/')